# Week 2 - Day 2: Parallel work and Orchestrator-Worker

> **How this notebook works.** Each **TODO** cell is real code with a few blanks: `___`. Replace every `___` (the comment on that line tells you what goes there), run the cell, then run the **check** below it and compare what it prints with the `# expected:` comments. After each part there is **Classwork**: a new problem that you build **yourself**, from an empty cell. Use the code above as your guide, but type every line yourself; open the hint only if you are stuck. Finished early? Try the *If you finish early* idea under it.

On Day 1 the steps ran **one after another**. Today steps run **at the same time**:
1. **Parallelization**: a few fixed steps run side by side, then one step joins their results.
2. **Orchestrator-worker**: a **planner** splits a big question into small questions, LangGraph starts **one worker for each**, and a writer joins their notes into a short brief.

Both patterns are in the LangGraph guide **[Workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents)**.

## Setup
You need **two** free keys, the same as Week 1:
- **Groq** (console.groq.com -> API Keys), saved as `GROQ_API_KEY`
- **Tavily** (tavily.com), saved as `TAVILY_API_KEY`

**On Google Colab:** click the key icon (Secrets) on the left, add both keys, and switch **Notebook access** on. **On your computer:** put them in a `.env` file.

> **Tips.** The free Groq plan allows about 8,000 tokens a minute. If a cell pauses, it is waiting for the limit. If a cell ends with `RateLimitError`, wait a minute and run it again. If a cell ends with `NameError: name '___' is not defined`, a blank is still empty. If a check ends with another `NameError`, run the TODO cell above it first.

In [ ]:
# What: installs the libraries for this notebook (about 1 minute, once per Colab session).
# Why:  Colab starts without LangChain, LangGraph or the Groq and Tavily connectors.
# How:  %pip is the notebook command for pip: -q keeps it quiet, -U gets the newest versions.
%pip install -qU langchain langgraph langchain-groq langchain-tavily python-dotenv

In [ ]:
# What: puts your API keys into the environment, where LangChain looks for them.
# Why:  keys are secrets: they never go into the notebook text, so the notebook can be shared safely.
# How:  on Colab, userdata reads the Secrets panel; on your computer, load_dotenv reads a .env file.
import os
from dotenv import load_dotenv

load_dotenv()                                                   # on your computer: reads the keys from a .env file
try:
    from google.colab import userdata                           # on Google Colab: reads the keys from Secrets
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
except ImportError:
    pass

print("Groq key found:", "GROQ_API_KEY" in os.environ)
print("Tavily key found:", "TAVILY_API_KEY" in os.environ)

In [ ]:
# What: the chat model for the whole week: gpt-oss-120b, hosted by Groq.
# Why:  one model object, created once, is used by every step below.
# How:  init_chat_model("provider:model") picks the right connector. temperature=0 gives the most repeatable answers.
#       max_retries=10: when the free plan's speed limit is reached, it waits and tries again instead of failing.
from langchain.chat_models import init_chat_model

llm = init_chat_model("groq:openai/gpt-oss-120b", temperature=0, max_retries=10)

from typing import Annotated                        # Annotated: attaches a reducer to a state key
import operator                                     # operator.add: the reducer that joins lists
from typing_extensions import TypedDict             # TypedDict: describes the keys of a state
from pydantic import BaseModel, Field               # BaseModel + Field: the classes for structured output
from langgraph.graph import StateGraph, START, END  # the graph builder, and its start and end points
from langgraph.types import Send                    # Send: starts one worker with its own input
from langchain_tavily import TavilySearch           # the ready-made web search tool
from langchain_core.callbacks import get_usage_metadata_callback   # the token meter
from IPython.display import Image, display          # shows the graph picture in the notebook

search = TavilySearch(max_results=2)          # the same web search tool as Day 1: 2 results per search

---
# Part 1 - Parallelization

Three nodes look at the same question **from three sides**. They do not need each other, so LangGraph runs them **at the same time**. A fourth node, the **aggregator**, joins the three answers.

```
        +--> find_pros  --+
START --+--> find_cons  --+--> aggregator --> END
        +--> find_costs --+
```

There is no new code for "parallel". You only draw the edges: **three edges from START**, and **three edges into the aggregator**. LangGraph waits for all three before the aggregator runs.

In [ ]:
# Given: the state and the three side nodes. They are the same, with a different prompt.
# What: a state with one key per side (pros, cons, costs), and the three side nodes.
# Why:  nodes that run at the same time must write to different keys, or one would overwrite another.
# How:  each node is one model call with its own prompt, and returns only its own key.
class ParallelState(TypedDict):
    question: str
    pros: str
    cons: str
    costs: str
    summary: str

def find_pros(state: ParallelState):
    msg = llm.invoke(f"List 3 short advantages for this question: {state['question']}")
    return {"pros": msg.content}

def find_cons(state: ParallelState):
    msg = llm.invoke(f"List 3 short disadvantages for this question: {state['question']}")
    return {"cons": msg.content}

def find_costs(state: ParallelState):
    msg = llm.invoke(f"List the main costs, in 3 short points, for this question: {state['question']}")
    return {"costs": msg.content}

In [ ]:
# TODO 1: the aggregator joins the three answers. Fill the 2 blanks.
# What: the aggregator joins the three answers into one short summary.
# Why:  the three sides are notes; the user wants one answer.
# How:  an f-string builds one text from the three keys (\n = new line), then one model call uses only that text.
def aggregator(state: ParallelState):
    text = f"Pros:\n{state['pros']}\n\nCons:\n{state['cons']}\n\nCosts:\n{state['costs']}\n\n"
    msg = llm.invoke(f"Using only these notes, answer in 4 sentences: {state['question']}\n\n" + ___)   # the joined notes
    return {___: msg.content}                                                                 # the key it fills

In [ ]:
# TODO 2: build the graph. Fill the 2 blanks.
# What: the parallel graph.
# Why:  there is no special "parallel" command: the edges decide what runs at the same time.
# How:  three edges from START = the three sides start together.
#       three edges into the aggregator = it waits until all three are done.
builder = StateGraph(ParallelState)
builder.add_node("find_pros", find_pros)
builder.add_node("find_cons", find_cons)
builder.add_node("find_costs", find_costs)
builder.add_node("aggregator", aggregator)
builder.add_edge(START, "find_pros")             # three edges from START: the three nodes start together
builder.add_edge(START, "find_cons")
builder.add_edge(START, ___)           # the third side node
builder.add_edge("find_pros", "aggregator")      # three edges into the aggregator: it waits for all three
builder.add_edge("find_cons", "aggregator")
builder.add_edge("find_costs", ___)    # where the third side goes
builder.add_edge("aggregator", END)
parallel_workflow = builder.compile()

In [ ]:
# Draw it: three boxes side by side between START and the aggregator.
display(Image(parallel_workflow.get_graph().draw_mermaid_png()))

In [ ]:
# CHECK 1
# How it works: one invoke runs all four nodes; the summary should use pros, cons and costs.
state = parallel_workflow.invoke({"question": "Should a student in Kathmandu buy an electric scooter?"})
print(state["summary"])         # expected: a 4-sentence answer that mentions pros, cons and costs

## Classwork 1 · Add a fourth side (about 12 minutes)
**Build it yourself** in the empty cell below. Use Part 1 as your guide, but type every line yourself.

Make a new parallel workflow with **four** sides: pros, cons, costs and **risks**. You may reuse `find_pros`, `find_cons` and `find_costs` from Part 1; you write the rest.

**Steps**
1. A state `FourState` with the keys `question`, `pros`, `cons`, `costs`, `risks` and `summary`.
2. A node `find_risks` that lists 3 risks and returns only `risks`.
3. A new aggregator that joins all **four** keys into a 4-sentence answer, saved in `summary`.
4. The graph: four edges from START, four edges into the aggregator, and the aggregator to END. Compile it as `four_sides`.

<details><summary><b>Stuck? Open a hint</b></summary>

Copy the shape of the Part 1 code. You add a key, a node and two edges: `builder.add_edge(START, "find_risks")` and `builder.add_edge("find_risks", "aggregator")`. A `for` loop over the four node names can add all eight edges in three lines.
</details>

In [ ]:
# CLASSWORK 1: build four_sides (state keys: question, pros, cons, costs, risks, summary).

# Write your code here.

In [ ]:
# CHECK
state = four_sides.invoke({"question": "Should our school replace paper books with tablets?"})
print(state["risks"])        # expected: 3 risks
print()
print(state["summary"])      # expected: 4 sentences that also mention a risk

**If you finish early:** draw `four_sides` with `draw_mermaid_png`. How many nodes run at the same time now?

---
# Part 2 - Orchestrator-worker

In Part 1 **you** decided the three steps. Often you do not know the steps in advance: they depend on the question. Then a **planner** (the orchestrator) writes the steps, and LangGraph starts **one worker for each step**.

```
                         +--> research_step ("sub-question 1") --+
START --> make_plan -----+--> research_step ("sub-question 2") --+--> write_brief --> END
                         +--> research_step ("sub-question 3") --+
```

First, the planner: structured output again, like the router of Day 1. This time the field is a **list** of steps.

In [ ]:
# TODO 3: the plan class and the planner. Fill the blank.
# What: a Plan class with a list of steps, and a planner model that must fill it.
# Why:  the next nodes need a real Python list, so the code can start one worker per step.
# How:  list[str] means "a list of texts". The description says how many steps and what each one looks like.
class Plan(BaseModel):
    steps: list[str] = Field(description="3 short web search questions. Each one names the topic, so it can be searched alone.")

planner = llm.with_structured_output(___, method="json_schema", strict=True)     # the class the model must fill

In [ ]:
# CHECK 2
# How it works: plan.steps is a normal Python list that you can loop over.
plan = planner.invoke("Is it worth learning Rust in 2026?")
print(plan.steps)               # expected: a list of about 3 short search questions

Now two new LangGraph pieces:
- **A reducer.** Every worker writes to the same key, `notes`. Normally each write **replaces** the last one. `Annotated[list, operator.add]` tells LangGraph to **add** each note to the list instead.
- **`Send`.** `Send("research_step", {"step": step})` starts **one worker** and gives it its own small input. You make one `Send` for each step of the plan.

In [ ]:
# TODO 4: the two states. Fill the blank.
# What: the main state, and the small state of one worker.
# Why:  all workers write to the same key, notes. Without a reducer, each write would REPLACE the last one.
# How:  Annotated[list, operator.add] says: when a node returns notes, ADD its list to the list already there.
#       WorkerState is the only input a worker gets: its own step.
class State(TypedDict):
    question: str
    steps: list[str]
    notes: Annotated[list, ___]     # the reducer: operator.add, so every worker ADDS its note
    brief: str

class WorkerState(TypedDict):                # the small input of one worker
    step: str

In [ ]:
# TODO 5: the three nodes. Fill the 2 blanks.
# What: the planner node, the worker node and the writer node.
# Why:  one job each: plan, research one step, join everything.
# How:  research_step calls the search tool directly (no agent: every worker needs exactly one search),
#       then one model call turns the results into a 2-sentence note. The note goes back inside a list,
#       so the reducer can add the lists together.
def make_plan(state: State):
    plan = planner.invoke(state["question"])
    return {"steps": plan.___}                            # the field of Plan

def research_step(state: WorkerState):                     # one copy of this node runs for each step
    results = search.invoke({"query": state["step"]})       # one web search
    msg = llm.invoke(f"Answer in 2 sentences: {state['step']}\nUse only these search results, and end with the link you used:\n{results}")
    return {"notes": [___]}                         # the note, inside a list: the reducer adds the lists together

def write_brief(state: State):
    notes = "\n\n".join(state["notes"])
    msg = llm.invoke(f"Write a brief answer of at most 120 words to: {state['question']}\nUse only these notes, and keep the links:\n{notes}")
    return {"brief": msg.content}

In [ ]:
# TODO 6: start one worker for each step. Fill the blank.
# What: the function that starts one worker for each step of the plan.
# Why:  the number of steps is only known when the plan exists, so these edges cannot be drawn in advance.
# How:  Send("research_step", {"step": step}) = run research_step once, with this input.
#       Returning a list of Sends starts all of them, at the same time.
def assign_workers(state: State):
    workers = []
    for step in state["steps"]:
        workers.append(Send(___, {"step": step}))      # the name of the worker node
    return workers

In [ ]:
# TODO 7: build the graph. Fill the 2 blanks.
# What: the orchestrator-worker graph.
# How:  after make_plan, assign_workers returns the Sends. ["research_step"] tells LangGraph where Sends can go
#       (it needs this to draw the graph). write_brief runs once, after every worker has finished.
builder = StateGraph(State)
builder.add_node("make_plan", make_plan)
builder.add_node("research_step", research_step)
builder.add_node("write_brief", write_brief)
builder.add_edge(START, "make_plan")
builder.add_conditional_edges("make_plan", ___, ["research_step"])     # the function that makes the Sends
builder.add_edge("research_step", ___)                                   # write_brief waits for all the workers
builder.add_edge("write_brief", END)
research_workflow = builder.compile()

In [ ]:
# Draw it: make_plan -> research_step (as many as the plan says) -> write_brief.
display(Image(research_workflow.get_graph().draw_mermaid_png()))

In [ ]:
# CHECK 3: the whole workflow. It takes about 30 seconds.
# How it works: one note per step (thanks to the reducer), and a brief with links. The meter counts every call.
with get_usage_metadata_callback() as cb:
    state = research_workflow.invoke({"question": "Is it worth learning Rust in 2026?"})

print("Plan :", state["steps"])
print("Notes:", len(state["notes"]))      # expected: the same number as the steps in the plan
print()
print(state["brief"])                     # expected: a short brief with links
print()
print("Tokens:", cb.usage_metadata)

## Classwork 2 · A trip planner (about 22 minutes)
**Build it yourself** in the two empty cells below: an orchestrator-worker workflow **without web search**. Use Part 2 as your guide, but type every line yourself.

```
                       +--> plan_day ("Day 1: ...") --+
START --> split_trip --+--> plan_day ("Day 2: ...") --+--> join_days --> END
                       +--> plan_day ("Day 3: ...") --+
```

**Steps**
1. **2a:** a class `Days` with one field, `days`: a list of texts, one short line per day. Make `day_planner` with structured output, and run its check.
2. **2b:** a state `TripState` with `trip`, `days`, `day_plans` (with the reducer `operator.add`) and `plan`, and a worker state `DayState` with `day`.
3. Three nodes: `split_trip` (asks `day_planner`, saves `days`), `plan_day` (one model call: morning, afternoon and evening of one day, returned **inside a list**), and `join_days` (joins the plans in day order and adds a budget tip).
4. A function `send_days` that returns one `Send("plan_day", {"day": day})` for each day.
5. The graph: START, `split_trip`, the workers, `join_days`, END. Compile it as `trip_planner`.

<details><summary><b>Stuck? Open a hint</b></summary>

- 2a is like `Plan` in TODO 3.
- The states are like TODO 4: `day_plans: Annotated[list, operator.add]`.
- `plan_day` returns `{"day_plans": [msg.content]}`: the plan inside a list.
- `send_days` is like `assign_workers` in TODO 6, and the graph is like TODO 7.
</details>

In [ ]:
# CLASSWORK 2a: the class Days and the model day_planner.

# Write your code here.

In [ ]:
# CHECK 2a
print(day_planner.invoke("A 3-day trip to Pokhara for two students on a small budget").days)     # expected: 3 lines, one for each day

In [ ]:
# CLASSWORK 2b: TripState, DayState, the three nodes, send_days, and the graph trip_planner.

# Write your code here.

In [ ]:
# CHECK 2b: about 20 seconds.
state = trip_planner.invoke({"trip": "A 3-day trip to Pokhara for two students on a small budget"})
print("Days      :", len(state["days"]))          # expected: 3
print("Day plans :", len(state["day_plans"]))     # expected: 3, one from each worker
print()
print(state["plan"])                              # expected: one plan, Day 1 to Day 3, and a budget tip

**If you finish early:** ask for a **5-day** trip. How many workers run now? Put the run inside `with get_usage_metadata_callback() as cb:` and compare the tokens with the 3-day trip.

> **If you hit the speed limit:** the workers run at the same time, and together they can pass the free plan's 8,000 tokens a minute. Then run `research_workflow.invoke({"question": "..."}, {"max_concurrency": 1})`: only one worker runs at a time.

**Talk about it:** when is a planner better than the fixed steps of Part 1? Think of a question where you cannot know the steps in advance.

---
# Today you learned
- **Parallelization**: draw several edges from one node, and the nodes run at the same time. A node with several edges **into** it waits for all of them.
- **Orchestrator-worker**: a planner writes the steps, and **`Send`** starts one worker for each step.
- **A reducer** (`Annotated[list, operator.add]`) adds the notes of all the workers to one list.

**Next class: the evaluator-optimizer pattern.** One model writes, another grades it, and the graph loops until the answer is good.